# Trial Thresholds — grid T x K GraphRAG 2-Stage, dan GraphRAG 1-Stage di tiap T

Notebook ini mencari **threshold tabel (T) dan kolom (K)**. Preprocessing 2-Stage diatur dengan **saklar on/off** di bagian 3; defaultnya preprocessing `config.py` (poin 30). GraphRAG 1-Stage (nama tabel saja, poin 39) dijalankan di tiap T sebagai pembanding. Sampel 20% dev set per tingkat kesulitan (stratified).

| Bagian | Isi |
|---|---|
| 1-4 | Setup, import, konfigurasi (saklar preprocessing, grid), data |
| 5 | Index embedding |
| 6 | **Grid**: SLA, kolom, dan token tiap (T, K), dan 1-Stage tiap T |
| 7 | **EX** tiap baris (zero-shot, butuh GPU) |
| 8 | **2-Stage vs 1-Stage di T yang sama**, lalu threshold dipilih **manual** oleh peneliti (`BEST_T` / `BEST_K`) |
| 9 | Bandingkan dua baris per query |
| 10 | Export Excel (`Trial Thresholds`) |

Efek tiap langkah preprocessing ada di `trial_preprocessing.ipynb`. Kode bersama kedua notebook ada di `src/experiments/trial_steps.py`.

> Notebook ini hanya menjalankan eksperimen. Threshold dipilih peneliti dari tabel bagian 6-8, lalu ditulis ke `config.py` (poin 33).

## 1. Setup

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
REPO_BRANCH = "main"

def sh(cmd: str) -> None:
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

def is_repo_root(p: Path) -> bool:
    return (p / "src" / "core" / "config.py").is_file()

if not is_repo_root(Path.cwd()):
    if is_repo_root(Path.cwd().parent):      # dijalankan dari notebooks/
        os.chdir("..")
    else:                                    # Kaggle: clone dulu
        if not is_repo_root(Path("thesis-2803").resolve()):
            sh(f"git clone -b {REPO_BRANCH} {REPO_URL}")
        else:                                # sudah pernah di-clone: pastikan di branch yang benar
            sh(f"git -C thesis-2803 fetch origin {REPO_BRANCH} && git -C thesis-2803 checkout {REPO_BRANCH} && git -C thesis-2803 pull origin {REPO_BRANCH}")
        os.chdir("thesis-2803")
REPO_ROOT = Path.cwd()
assert is_repo_root(REPO_ROOT), f"{REPO_ROOT} bukan root repo thesis-2803"
print("Repo root:", REPO_ROOT)
print("Branch   :", subprocess.run("git rev-parse --abbrev-ref HEAD", shell=True, capture_output=True, text=True).stdout.strip())

# nltk dibutuhkan parser SQL Spider (process_sql.py); sentence-transformers untuk BGE-M3
sh("pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers nltk openpyxl")
import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"[WARN] nltk.download({pkg}) gagal: {e}")

## 2. Import

In [ ]:
import sys, json, time, shutil, subprocess
from collections import Counter
from dataclasses import replace

import numpy as np
import pandas as pd
import networkx as nx
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sentence_transformers import util
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_ROOT))

from src.core.config import PipelineConfig
from src.core.schema import normalize_identifier
from src.retrieval.retrieval import (
    retrieve_candidate_tables, retrieve_graphrag_schema, column_nodes_to_schema, _extract_phrases, k_from_pct,
    value_linked_columns, trace_schema_paths, traversal_graph,
)
from src.retrieval.value_linking import get_value_index
from src.experiments import trial_steps as ts
from src.experiments import result_tables as rt
from src.experiments.result_tables import show
from src.metrics.sla import compute_sla, aggregate_sla
from src.utils.sampling import distribution_table

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Imports OK | CUDA:", torch.cuda.is_available())

## 3. Konfigurasi

**Saklar preprocessing 2-Stage** (`True` = nyala). Default = preprocessing `config.py` (poin 30). Kalau saklar diubah, notebook mencetak bedanya dengan `config.py`; preprocessing yang dipakai pipeline tetap `config.py` sampai diubah di sana.

| Saklar | `True` | `False` |
|---|---|---|
| `STOPWORD_FILTER` | buang n-gram yang diawali/diakhiri stopword (`edge`) | semua n-gram (`none`) |
| `VALUE_LINKING` | kolom yang nilainya disebut persis di pertanyaan ikut jadi anchor | mati |
| `VALUE_NUMBERS` | value linking juga untuk angka/tahun (hanya kalau `VALUE_LINKING = True`) | teks saja |
| `FK_ONLY` | dari jalur traversal hanya kolom JOIN (`fk_only`) | semua node di jalur (`full`) |
| `CONNECT_NO_PK` | kolom tabel tanpa PK disambung di graph traversal | tidak |
| `NATURAL_NAMES` | embed nama natural Spider (`student id`) | nama asli (`stuid`) |

`EXTRA_PREPROC` menambah preprocessing lain ke grid yang sama: tiap entri hanya menyebut saklar yang berbeda dari di atas. Kosong = hanya satu preprocessing.

| Knob lain | Arti |
|---|---|
| `GRID_T`, `GRID_K` | threshold yang dicoba. Tiap (T, K) x preprocessing = satu run; 1-Stage hanya punya T |
| `RUN_ONE_STAGE` | 1-Stage di tiap T (pembanding bagian 8) |
| `RUN_EX` | EX zero-shot tiap baris (bagian 7, butuh GPU) |
| `TARGET_P`, `TARGET_R` | baris yang memenuhi keduanya diberi warna hijau (penanda visual saja) |

In [ ]:
SAMPLE_RATIO = 0.20        # 20% per tingkat kesulitan, sama dengan ablation few-shot
MAX_NGRAM = 2              # bigram (config.py)

# --- Saklar preprocessing GraphRAG 2-Stage (True = nyala); default = config.py ---
STOPWORD_FILTER = True     # True = edge, False = none
VALUE_LINKING   = True     # value linking teks
VALUE_NUMBERS   = False    # value linking juga untuk angka/tahun
FK_ONLY         = True     # True = fk_only, False = full
CONNECT_NO_PK   = True     # sambung tabel tanpa PK di graph traversal
NATURAL_NAMES   = True     # embed nama natural Spider
# Preprocessing tambahan di grid yang sama; tiap entri = saklar yang berbeda dari di atas
EXTRA_PREPROC = []         # contoh: [dict(STOPWORD_FILTER=False, VALUE_NUMBERS=True)]

GRID_T = (0.50, 0.55, 0.60, 0.65, 0.70)    # threshold tabel (Stage 1), 2-Stage dan 1-Stage
GRID_K = (0.55, 0.60, 0.65, 0.70, 0.75)    # threshold kolom (Stage 2), hanya 2-Stage
RUN_ONE_STAGE = True       # 1-Stage (nama tabel saja) di tiap T
EXCLUDE_EMPTY_GOLD = False
RUN_EX = True              # bagian 7: EX zero-shot tiap baris (butuh GPU)
TARGET_P, TARGET_R = 0.50, 0.90

cfg = PipelineConfig(top_k_tables_pct=0, top_k_columns_pct=0, max_ngram=MAX_NGRAM)
cfg.data_path = ts.find_data_path(cfg, REPO_ROOT)
_switches = dict(STOPWORD_FILTER=STOPWORD_FILTER, VALUE_LINKING=VALUE_LINKING, VALUE_NUMBERS=VALUE_NUMBERS,
                 FK_ONLY=FK_ONLY, CONNECT_NO_PK=CONNECT_NO_PK, NATURAL_NAMES=NATURAL_NAMES)
PREPROC = [ts.switches_to_settings(**{k.lower(): v for k, v in {**_switches, **extra}.items()})
           for extra in [{}] + EXTRA_PREPROC]
_config_py = ts.settings_of(cfg)
print("Data path:", cfg.data_path)
for i, s in enumerate(PREPROC):
    diff = {k: v for k, v in s.items() if v != _config_py[k]}
    print(f"Preprocessing {i}: {ts.describe(s)}" + (f"  (beda dengan config.py: {diff})" if diff else "  (= config.py)"))
n_runs = len(PREPROC) * len(GRID_T) * len(GRID_K) + (len(GRID_T) if RUN_ONE_STAGE else 0)
print(f"{n_runs} baris di grid")

## 4. Load data, graph, sampel dev set, dan gold schema

In [ ]:
data = ts.load_trial_data(cfg, SAMPLE_RATIO, EXCLUDE_EMPTY_GOLD)
# Nama pendek untuk sel EDA dan analisis di bawah
schema_df, graph, db_schema, samples, sample_dbs = data.schema_df, data.graph, data.db_schema, data.samples, data.sample_dbs
print(f"Dev set: {len(data.dev_data)} query | sampel: {len(data.indices)} | dipakai: {len(samples)} "
      f"| gold gagal di-parse: {data.n_failed} | gold kosong: {data.n_empty}")
print(f"Database di sampel: {len(sample_dbs)} dari {schema_df['Database'].nunique()}")
display(pd.DataFrame(distribution_table(data.dev_data, data.indices)).style.format(
    {"full_share": "{:.1%}", "sample_share": "{:.1%}"}).set_caption("Distribusi difficulty: dev penuh vs sampel"))

## 5. Bangun index embedding

In [ ]:
data = ts.load_embedder(data)
for db in tqdm(sample_dbs, desc="Index"):
    for names in dict.fromkeys(s["nama_embed"] for s in PREPROC):
        ts.index_for(data, db, replace(cfg, **ts.knobs_of({"nama_embed": names})))
    if RUN_ONE_STAGE:
        ts.table_index_for(data, db, embed_columns=False)
print("Selesai.")

## 6. Grid: SLA, kolom, dan token

Satu baris = satu preprocessing di satu (T, K), atau 1-Stage di satu T. Makin rendah K: recall naik, precision turun. Rekap lengkap tersimpan di `outputs/tables/trial_thresholds_recap.csv` dan sheet Excel `Trial Thresholds`; tabel di bawah hanya kolom inti. Arti tiap kolom: `context/TRIAL_COLUMNS_EXPLAINED.md`.

In [ ]:
TRIALS = [ts.two_stage_trial(cfg, s, t, k) for s in PREPROC for t in GRID_T for k in GRID_K]
if RUN_ONE_STAGE:
    TRIALS += [ts.one_stage_trial(cfg, t) for t in GRID_T]
TRIAL_CACHE = globals().get("TRIAL_CACHE", {})     # hasil per query yang sudah dihitung, dipakai ulang kalau sel dijalankan lagi
recap, RESULTS = ts.run_trials(data, TRIALS, TRIAL_CACHE, progress=lambda x: tqdm(x, desc="Grid"))
print("Rekap ->", ts.save_recap(recap, REPO_ROOT / "outputs" / "tables", ts.THRESHOLDS_RECAP))
print(f"n = {len(samples)} query | MAX_NGRAM = {cfg.max_ngram}")
display(ts.styled_recap(recap, TARGET_P, TARGET_R))

## 7. EX per baris (zero-shot, butuh GPU)

- **Zero-shot:** k belum dipilih pada tahap ini, dan tanpa contoh few-shot beda EX hanya berasal dari beda schema. Prompt-nya sama persis dengan `prompt_tokens_k0`, jadi EX dan token di satu baris berasal dari prompt yang sama. Yang dibaca adalah **perbandingan antar konfigurasi**, bukan EX akhir.
- **Prompt yang identik hanya digenerate sekali** (greedy, deterministik) dan disimpan di `outputs/tables/sla_trial_ex_cache.jsonl`, dipakai bersama kedua notebook trial. Jalankan ulang sel ini untuk melanjutkan run yang terputus; di Kaggle, unduh file cache itu supaya tidak hilang.
- **Tidak memilih apa pun** (poin 33). EX dihitung di dev, jadi ikut dalam limitation poin 34.

In [ ]:
if RUN_EX:
    recap, llm = ts.score_ex(recap, RESULTS, cfg, data.tokenizer, REPO_ROOT / ts.EX_CACHE, globals().get("llm"),
                             progress=lambda x: tqdm(x, desc="EX"))
    ts.save_recap(recap, REPO_ROOT / "outputs" / "tables", ts.THRESHOLDS_RECAP)
display(ts.styled_recap(recap, TARGET_P, TARGET_R))

## 8. GraphRAG 2-Stage vs GraphRAG 1-Stage di T yang sama, lalu pilih threshold (manual)

Sel ini **tidak memilih** threshold. Ia menyusun bahan keputusan, dan peneliti mengisi `BEST_T` / `BEST_K` di sel berikutnya.

Perbandingan apple-to-apple (IMPLEMENTATION_DECISIONS.md poin 33):
- **T sama untuk kedua pipeline.** GraphRAG 1-Stage tidak punya threshold kolom, jadi setiap baris GraphRAG 2-Stage `(T, K)` dibandingkan dengan **satu** baris `GraphRAG 1-Stage | T` yang sama. Mengganti K hanya mengubah sisi GraphRAG 2-Stage.
- **Query sama** (sampel yang sama), jadi reduksi = `1 - mean(GraphRAG 2-Stage) / mean(GraphRAG 1-Stage)` = `1 - Σ GraphRAG 2-Stage / Σ GraphRAG 1-Stage`.
- Tiga ukuran, dari yang paling dekat ke retrieval sampai yang dibayar LLM: **kolom** di prompt, **token schema** (DDL saja), **token prompt** (zero-shot). Reduksi token prompt selalu lebih kecil dari reduksi token schema, karena bagian non-schema (task, pertanyaan, instruksi, dan nanti few-shot) sama besar di kedua pipeline.
- **Dekomposisi kolom**: `table_effect` = 1 - kolom tabel GraphRAG 2-Stage kalau dikirim utuh / kolom GraphRAG 1-Stage (beda pilihan tabel); `prune_effect` = 1 - kolom GraphRAG 2-Stage / kolom tabel GraphRAG 2-Stage utuh (pemangkasan kolom). `(1 - total) = (1 - table_effect) x (1 - prune_effect)`.
- SLA dan EX (bagian 7, kalau `RUN_EX = True`) ditampilkan di samping supaya penghematan dibaca bersama recall dan EX yang dikorbankan.
- Perbandingan tiap baris dengan DDL penuh database-nya ada di rekap (kolom `*_db`, sheet `Trial Thresholds`).

In [ ]:
vs = ts.compare_with_one_stage(recap)
vs.to_csv(REPO_ROOT / "outputs" / "tables" / "trial_thresholds_vs_1stage.csv", float_format="%.4f")
_pct = [c for c in ["tbl_R", "col_P_ne", "col_R_ne", "col_R_ne_B", "cols_red", "schema_tok_red", "prompt_tok_red",
                    "table_effect", "prune_effect", "ex_G", "ex_B", "ex_delta"] if c in vs.columns]
display(vs.style.format({**{c: "{:.1%}" for c in _pct}, "table_threshold": "{:.2f}", "column_threshold": "{:.2f}",
                         **{c: "{:.1f}" for c in vs.columns if c.endswith(("_G", "_B")) and c not in _pct}}))
print("*_G = 2-Stage, *_B = 1-Stage di T yang sama, *_red = 1 - G/B (positif = 2-Stage lebih hemat). Index = preprocessing + T/K")

In [ ]:
# PILIHAN PENELITI: isi setelah membaca tabel bagian 6-8. None = tetap pakai threshold config.py.
BEST_T = None        # threshold tabel, dipakai 2-Stage DAN 1-Stage
BEST_K = None        # threshold kolom, hanya 2-Stage
BEST_PREPROC = 0     # nomor preprocessing yang dicetak di bagian 3 (0 = saklar utama)

_out = REPO_ROOT / "outputs" / "tables"
if BEST_T is not None and BEST_K is not None:
    settings = PREPROC[BEST_PREPROC]
    trial = ts.two_stage_trial(cfg, settings, BEST_T, BEST_K)[0]
    rec = {"table_similarity_threshold": BEST_T, "semantic_similarity_threshold": BEST_K, "preprocessing": settings,
           "preprocessing_knobs": ts.knobs_of(settings), "rule": "dipilih manual oleh peneliti", "n": len(samples),
           "max_ngram": cfg.max_ngram}
    if trial in vs.index:
        rec.update({k: float(v) for k, v in vs.loc[trial].items() if k not in ("table_threshold", "column_threshold")})
    (_out / "best_thresholds.json").write_text(json.dumps(rec, indent=2), encoding="utf-8")
    print(f"Dipilih: T = {BEST_T:.2f}, K = {BEST_K:.2f}, {ts.describe(settings)} -> {_out / 'best_thresholds.json'}\n"
          "Tulis ke src/core/config.py (atau isi di eval_pipeline.ipynb / ablation_fewshot.ipynb):\n"
          f"    table_similarity_threshold: float = {BEST_T:.2f}\n    semantic_similarity_threshold: float = {BEST_K:.2f}")
    if settings != ts.settings_of(cfg):
        print(f"Preprocessing ini beda dengan config.py (poin 30): ubah juga {ts.knobs_of(settings)}")
else:
    print(f"BEST_T / BEST_K belum diisi (config.py: T = {cfg.table_similarity_threshold:.2f}, K = {cfg.semantic_similarity_threshold:.2f})")

## 9. Bandingkan dua baris per query

Isi `A` dan `B` dengan nama baris (index tabel rekap). Hasilnya: kolom yang ditambahkan B (dan kolom gold yang ikut masuk), serta kolom yang dibuang B (dan kolom gold yang ikut hilang).

In [ ]:
names = list(recap.index)
A, B = names[0], names[1]
d = ts.compare_trials(RESULTS, A, B)
print(f"{A}\n  ->  {B}\nquery berubah: {len(d)} / {len(samples)} | gold masuk: {(d['gold_gained'].str.len() > 0).sum()} "
      f"| gold hilang: {(d['gold_lost'].str.len() > 0).sum()}")
display(d[d["gold_gained"].str.len() > 0][["qid", "db_id", "question", "gold_gained", "added"]].head(10))
display(d[d["gold_lost"].str.len() > 0][["qid", "db_id", "question", "gold_lost", "removed"]].head(10))

## 10. Export ke Excel

Sheet `Trial Thresholds`. Kalau `trial_preprocessing_recap.csv` juga ada di folder yang sama, sheet `Trial Preprocessing` dan `Trial Effects` ikut. Di Kaggle, salinannya ditaruh di `/kaggle/working/`, bersama cache EX.

In [ ]:
XLSX = REPO_ROOT / "outputs" / "tables" / "hasil_trial_thresholds.xlsx"
subprocess.run([sys.executable, "src/experiments/export_excel.py", "--skip-missing",
                "--tables-dir", str(REPO_ROOT / "outputs" / "tables"), "--out", str(XLSX)], check=True)
print("Excel:", XLSX)
if Path("/kaggle/working").exists():
    shutil.copy(XLSX, "/kaggle/working/hasil_trial_thresholds.xlsx")
    shutil.copy(REPO_ROOT / ts.EX_CACHE, "/kaggle/working/sla_trial_ex_cache.jsonl") if (REPO_ROOT / ts.EX_CACHE).exists() else None
    print("Salinan untuk diunduh: /kaggle/working/hasil_trial_thresholds.xlsx (dan cache EX)")